In [ ]:
#install modules
!pip install netmiko
!pip install PyYAML

import netmiko

from netmiko import ConnectHandler

class DeviceConnection:
    def __init__(self, device_info: dict):
        self.device_info = device_info
        self.connection = None

    def connect(self):
        self.connection = ConnectHandler(**self.device_info)
        self.connection.enable()

    def execute(self, command):
      if isinstance(command, list):
            # Send multiple config commands in configuration mode
          output = self.connection.send_config_set(command)
      else:
            # Send a single operational (show) command
          output = self.connection.send_command(command)
      return output

    def save_configuration(self):
        return self.connection.send_command("write memory")

    def disconnect(self):
        if self.connection:
            self.connection.disconnect()
            self.connection = None

import difflib
import time

def ping(conn):
  toPing=input('Enter the ping of the device:  ')
  output = conn.execute(f"ping {toPing}")
  return output

def pingAll(data, conn):
  outputArr=[]
  for device in data["devices"]:
    output = conn.execute(f"ping {data['devices'][device]['connection']['host']}")
    time.sleep(5)
    outputArr.append(output)
  return outputArr

def enable(device, conn): #not sure if needed
  return conn.execute(f"enable {device['connection']['secret']}")

def setInterfaces(devNo, conn, devices_list):
  for interface in devices_list[int(devNo)]['interfaces']:
    conn.execute(f"interface {interface}")
    conn.execute(f"ip address {interface['ip']} {interface['mask']}")

def setInterfacesV6(devNo, conn, devices_list):
      for interface in devices_list[int(devNo)]['interfaces']:
        conn.execute(f"interface {interface}")
        conn.execute(f"ipv6 address {interface['ipv6']}{interface['maskV6']}")


def showRun(conn):
  output = conn.execute(f"show run")
  return output


def vlan(conn):
  vlan_id = input('VLAN ID: ')
  vlan_name = input('VLAN Name: ')
  conn.execute([f'vlan {vlan_id}', f'name {vlan_name}'])
  output =  conn.execute([f'interface vlan {vlan_id}', 'ip address 192.168.10.1 255.255.255.0', 'no shutdown'])
  return output


def staticRoute(conn):
  destination_ip = input('Destination IP: ')
  mask_ip = input('Mask IP: ')
  next_hop_ip = input('Next Hop IP: ')
  return conn.execute([f'ip route {destination_ip} {mask_ip} {next_hop_ip}'])

def backupConfBefore(conn):

  running_config = conn.execute("show running-config")
  hostname_line = [line for line in running_config.splitlines() if line.strip().startswith('hostname ')]
  if hostname_line:
      hostname = hostname_line[0].split(' ', 1)[1].strip()
      filename = f"{hostname}_configBefore.txt"
      with open(filename, 'w') as f:
          f.write(running_config)
      print(f"Running configuration saved to {filename}")
  else:
      print("Could not determine hostname from running configuration.")

def backupConfAfter(conn):

  running_config = conn.execute("show running-config")
  hostname_line = [line for line in running_config.splitlines() if line.strip().startswith('hostname ')]
  if hostname_line:
      hostname = hostname_line[0].split(' ', 1)[1].strip()
      filename = f"{hostname}_configAfter.txt"
      with open(filename, 'w') as f:
          f.write(running_config)
      print(f"Running configuration saved to {filename}")
  else:
      print("Could not determine hostname from running configuration.")

def compare_configs(file1, file2):
    try:
        with open(file1, 'r') as f1, open(file2, 'r') as f2:
            lines1 = f1.readlines()
            lines2 = f2.readlines()

        diff = difflib.unified_diff(lines1, lines2, fromfile=file1, tofile=file2)

        print(f"Differences between {file1} and {file2}:")
        for line in diff:
            print(line, end='')

    except FileNotFoundError:
        print("Error: One or both files not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

def acl(conn):
  acl_id = input('ACL ID: ')
  num_rules = int(input('Number of rules: '))
  acl_rules = []
  for i in range(num_rules):
      rule = input(f'Rule {i+1}: ')
      acl_rules.append(rule)
  acl_interface = input('Interface: ')
  acl_direction = input('Direction (in/out): ')


  outputArr = []
  out1 = conn.execute([f"access-list {acl_id} {r}" for r in acl_rules])
  out2 = conn.execute([f"interface {acl_interface}", f"ip access-group {acl_id} {acl_direction}"])

  outputArr.append(out1)
  outputArr.append(out2)

  return outputArr


import yaml
with open("devices_data.yaml") as file:
    data = yaml.safe_load(file)
    devices_list = list(data["devices"].keys())

while True:
    try:
        print('\n Available Devices:')
        print('\n Press 0 to exit')
        for i, device in enumerate(devices_list):
            print(f'{i+1}. {device}')

        devNo_input = input('Choose the number of the device: ')

        if not devNo_input.isdigit():
            print("Invalid input. Please enter a number.")
            continue

        devNo = int(devNo_input)

        if devNo == 0:
            break
        elif devNo < 0 or devNo > len(devices_list):
            print("Invalid device number. Please choose a number from the list.")
            continue
        else:
            devNo = devNo - 1
            conn = DeviceConnection(data["devices"][devices_list[devNo]]['connection'])
            conn.connect()
            backupConfBefore(conn)

            if data['devices'][devices_list[devNo]]['real_type'] == 'switchL2':
                print('\n1. Create VLAN\n2. Ping\n3. Compare Logs\n4. Ping All devices\n5. Save Config\n6. Exit')
                ch = input('Choose Action: ')

                if ch == '1':
                    print(vlan(conn))
                elif ch == '2':
                    print(ping(conn))
                elif ch == '3':
                     backupConfAfter(conn)
                     compare_configs(f'{devices_list[devNo]}_configBefore.txt', f'{devices_list[devNo]}_configAfter.txt')
                elif ch == '4':
                    print(pingAll(data, conn))
                elif ch == '5':
                    print(conn.save_configuration())
                elif ch == '6':
                    break

            elif data['devices'][devices_list[devNo]]['real_type'] == 'router':
                print('\n1. Ping\n2. Compare Logs\n3. Ping All devices\n4. Save Config\n5. Conf Static Route\n6. Conf ACL\n7 Exit')
                ch = input('Choose Action: ')

                if ch == '1':
                    print(ping(conn))
                elif ch == '2':
                     backupConfAfter(conn)
                     compare_configs(f'{devices_list[devNo]}_configBefore.txt',
                                     f'{devices_list[devNo]}_configAfter.txt')
                elif ch == '3':
                    print(pingAll(data, conn))
                elif ch == '4':
                    print(conn.save_configuration())
                elif ch == '5':
                    print(staticRoute(conn))
                elif ch == '6':
                    print(acl(conn))
                elif ch == '7':
                    break

            backupConfAfter(conn)

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals() and conn.connection:
            conn.disconnect()